# Day 1 · Hands-on 1 — Sampling variability
**About 30 min · work in pairs** (one person drives the keyboard, the other reads, predicts and questions — swap halfway)


In real research you see **one sample** and have to guess about the population. Here we cheat: we treat all 686 patients in the GBSG2 breast cancer trial as the *whole population*, so we know the true answer and can watch what samples do.



> **Two AI roles today — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to change, fix or explain *code*.
> - 🎓 **Claude / ChatGPT (other tab) = tutor you must check.** Ask it to *interpret*, then score its answer: **correct · complete · specific · actionable**.


**How this notebook works:** run cells from top to bottom. Each code cell has exactly one line marked `# 🔧 CHANGE THIS` — that is the only thing you need to edit. Lines starting with 📋 are results to enter in the shared sheet.

In [ ]:
# ▶️ Run this cell first (about 20 seconds). Nothing to change here.
%pip install -q lifelines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines.datasets import load_gbsg2

df = load_gbsg2()
print(f"✅ Loaded GBSG2: {len(df)} patients, {df.shape[1]} variables")

## A. Meet the population
### 🧭 Concept
`tsize` is tumour size in mm. Look at its shape: is it symmetric? Where is the mean compared with the bulk of the patients?

In [ ]:
# 🔧 CHANGE THIS: variable to look at — try "tsize", "age" or "pnodes"
variable = "tsize"

population = df[variable]
true_mean = population.mean()

plt.figure(figsize=(7, 3.5))
plt.hist(population, bins=30, color="lightgray", edgecolor="black")
plt.axvline(true_mean, color="red", linewidth=2, label=f"Population mean = {true_mean:.1f}")
plt.xlabel(variable); plt.ylabel("Number of patients"); plt.legend()
plt.title(f"The 'population': all {len(population)} GBSG2 patients")
plt.show()
print(f"Population mean of {variable}: {true_mean:.2f}")
print(f"Population SD of {variable}:   {population.std(ddof=0):.2f}")

## B. One study = one sample
### 🧭 Concept
A study recruits *n* patients at random. Its mean tumour size is an **estimate** of the population mean.

✍️ **Predict first:** if you run the cell below five times, how different will the five means be? Write your guess down, then run it five times and note the means.

In [ ]:
# 🔧 CHANGE THIS: number of patients in your study
n = 20

population = df["tsize"]
sample = population.sample(n=n, replace=True)   # random patients, as if the population were very large

print(f"Your study of {n} patients: mean tumour size = {sample.mean():.1f} mm")
print(f"True population mean:                  {population.mean():.1f} mm")

## C. Thousands of studies
### 🧭 Concept
Instead of running the study five times, the computer runs it **2,000 times** and plots all 2,000 means. This histogram is the *sampling distribution of the mean*. Its spread tells you how much one study's answer can wander.

Use the sample size your facilitator assigned to your pair.

In [ ]:
# 🔧 CHANGE THIS: sample size — use the value assigned to your pair
n = 20

n_studies = 2000
rng = np.random.default_rng()
population = df["tsize"].to_numpy()
true_mean = population.mean()

# 2,000 studies, each with n random patients; keep each study's mean
sample_means = rng.choice(population, size=(n_studies, n), replace=True).mean(axis=1)
spread = sample_means.std()

plt.figure(figsize=(7, 3.5))
plt.hist(sample_means, bins=40, color="steelblue", edgecolor="white")
plt.axvline(true_mean, color="red", linewidth=2, label="Population mean")
plt.xlim(5, 60)
plt.xlabel("Mean tumour size in one study (mm)"); plt.ylabel("Number of studies")
plt.title(f"2,000 studies of n = {n} patients"); plt.legend()
plt.show()

print("📋 For the shared sheet:")
print(f"   n = {n}   |   spread of the study means (SD) = {spread:.2f} mm")

### ✅ Concept check
1. Tumour sizes are skewed to the right. Is the histogram of *study means* skewed too? What changes as *n* grows?
2. If you quadruple *n* (e.g. 20 → 80), roughly what happens to the spread? (Wait for the shared sheet before you answer.)

<details><summary>Show answers</summary>

1. The means look much more symmetric and bell-shaped than the raw data, more so with larger *n* — this is the central limit theorem at work.
2. The spread roughly **halves**: it shrinks with √n, not with n. Four times the patients buys twice the precision.
</details>

## D. After the debrief: the √n rule
### 🧭 Concept
The spread of study means is called the **standard error (SE)**. Theory says SE = σ / √n, where σ is the population SD. The cell below simulates many sample sizes and overlays the formula. Try other variables: does the rule still hold?

In [ ]:
# 🔧 CHANGE THIS: variable — try "tsize", "age" or "pnodes"
variable = "tsize"

rng = np.random.default_rng()
population = df[variable].to_numpy()
sigma = population.std()
sizes = [5, 10, 20, 40, 80, 160, 320]

simulated = [rng.choice(population, size=(2000, n), replace=True).mean(axis=1).std() for n in sizes]
formula = [sigma / np.sqrt(n) for n in sizes]

plt.figure(figsize=(7, 3.5))
plt.plot(sizes, simulated, "o", markersize=8, label="Simulated spread of study means")
plt.plot(sizes, formula, "-", label="σ / √n")
plt.xlabel("Sample size n"); plt.ylabel("Standard error"); plt.legend()
plt.title(f"Standard error of the mean of {variable}")
plt.show()
for n, s, f in zip(sizes, simulated, formula):
    print(f"n = {n:4d}   simulated SE = {s:6.2f}   σ/√n = {f:6.2f}")

## 🤖 AI prompts
**🧮 Gemini in Colab (calculator):** select the cell in section C and ask:
> *Also print the 2.5th and 97.5th percentiles of `sample_means`.*

Check: does the new code use `sample_means`? Is the range about ±2 × the spread around the population mean?

**🎓 Claude / ChatGPT (tutor):** paste this, then score the answer with the four-point rubric:
> *I measured tumour size in 20 patients. The sample mean is 31 mm and the sample standard deviation is 14 mm. What is the difference between the standard deviation and the standard error here, and which one tells me how precise my estimate of the mean is?*

Look for: SD describes patients, SE describes the *estimate*; SE = 14/√20 ≈ 3.1 mm; SE shrinks with more patients but SD does not.

## 💬 Discuss with your partner
- A colleague reports "mean ± SD" for one experiment and "mean ± SE" for another. Why can the second look far more precise even with similar data?
- Your lab can afford 40 mice instead of 10. How much more precise will your mean be?
- What assumption did we make by sampling patients **at random**? When would a real study break it?